# Projeto de Machine Learning – Análise e Predição de Inadimplência em Crédito
## Metodologia: CRISP-DM

Este projeto tem como objetivo analisar e prever a inadimplência de clientes a partir de dados financeiros, utilizando modelos de aprendizado supervisionado.

### Dataset: Lending Club (2007-2018)
Plataforma americana de crédito peer-to-peer (P2P) que conecta investidores a pessoas físicas interessadas em obter financiamento.

### Variável Alvo:
- **0** → Cliente adimplente (pagou o empréstimo)
- **1** → Cliente inadimplente (não pagou)

In [ ]:
# CRISP-DM Fase 1 e 2 - Entendimento do Negócio e dos Dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, recall_score, roc_auc_score, roc_curve
)
from xgboost import XGBClassifier

print("Bibliotecas carregadas ✔")

## CRISP-DM Fase 2 — Entendimento dos Dados

O dataset contém mais de 2 milhões de registros e 151 colunas.
Foram selecionadas **13 features** relevantes para o problema de inadimplência.

### Features selecionadas:
| Feature | Descrição |
|---------|-----------|
| loan_amnt | Valor do empréstimo solicitado |
| term | Prazo do empréstimo (36 ou 60 meses) |
| int_rate | Taxa de juros |
| installment | Valor da parcela mensal |
| annual_inc | Renda anual do cliente |
| dti | Índice de endividamento |
| fico_range_high | Score de crédito máximo |
| revol_util | Utilização do crédito rotativo |
| delinq_2yrs | Nº de inadimplências nos últimos 2 anos |
| open_acc | Nº de contas abertas |
| pub_rec | Registros públicos negativos |
| mort_acc | Nº de contas de hipoteca |
| loan_status | Status do empréstimo |

In [ ]:
# CRISP-DM Fase 2 - Carregamento dos dados
dados = pd.read_csv(
    '/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz',
    compression='gzip',
    low_memory=False,
    usecols=[
        'loan_amnt', 'term', 'int_rate', 'installment',
        'annual_inc', 'dti', 'fico_range_high', 'revol_util',
        'delinq_2yrs', 'open_acc', 'pub_rec', 'mort_acc',
        'loan_status'
    ]
)
print(f"Shape original: {dados.shape}")
dados.head()

In [ ]:
print(dados.info())
print(dados.describe())

In [ ]:
print(dados['loan_status'].value_counts())

## CRISP-DM Fase 3 — Preparação dos Dados

### Decisões tomadas:
1. **Amostragem:** 200k registros para otimizar o processamento mantendo representatividade
2. **Filtro loan_status:** mantidos apenas `Fully Paid` e `Charged Off` (resultados finais conhecidos)
3. **Definição do target:** `Charged Off` = inadimplência. `Default` removido por baixa representatividade
4. **Remoção de nulos:** ~3,6% dos dados removidos (proporção baixa, não justifica imputação)
5. **Tratamento de outliers:** Winsorização via IQR para limitar valores extremos sem remover dados

In [ ]:
# Amostragem de 200k registros
dados = dados.sample(200000, random_state=42)
print(f"Shape após amostragem: {dados.shape}")

# Filtro apenas status definitivos
dados = dados[dados['loan_status'].isin(['Fully Paid', 'Charged Off'])]
print(f"Shape após filtro: {dados.shape}")
print(dados['loan_status'].value_counts())

In [ ]:
# Criar variável target
dados['inadimplente'] = (dados['loan_status'] == 'Charged Off').astype(int)

# Renomear colunas
rename_dict = {
    'loan_amnt': 'valor_emprestimo',
    'term': 'prazo',
    'int_rate': 'taxa_juros',
    'installment': 'parcela',
    'annual_inc': 'renda_anual',
    'dti': 'indice_endividamento',
    'fico_range_high': 'score_credito',
    'revol_util': 'uso_credito_rotativo',
    'delinq_2yrs': 'inadimplencias_2anos',
    'open_acc': 'contas_abertas',
    'pub_rec': 'registros_negativos',
    'mort_acc': 'contas_hipoteca'
}
dados = dados.rename(columns=rename_dict)

# Converter prazo
dados['prazo'] = dados['prazo'].str.replace(' months', '').astype(int)

print(dados.head())

In [ ]:
print("Nulos por coluna:")
print(dados.isnull().sum())

total_antes = len(dados)  # capturado antes do dropna — evita hardcode
print(f"\nTotal antes: {total_antes}")

dados = dados.dropna()
dados = dados.drop(columns=['loan_status'], errors='ignore')

total_depois = len(dados)
linhas_removidas = total_antes - total_depois
percentual_removido = (linhas_removidas / total_antes) * 100

print(f"Total após dropna: {total_depois}")
print(f"Linhas removidas: {linhas_removidas} ({percentual_removido:.2f}%)")

print("\nInformações do dataset após limpeza:")
print(dados.info())

In [ ]:
# Tratamento de outliers via IQR para colunas contínuas
colunas_iqr = ['renda_anual', 'indice_endividamento']

for col in colunas_iqr:
    Q1 = dados[col].quantile(0.25)
    Q3 = dados[col].quantile(0.75)
    IQR = Q3 - Q1
    dados[col] = dados[col].clip(
        lower=Q1 - 1.5 * IQR,
        upper=Q3 + 1.5 * IQR
    )

# Para colunas com forte concentração em zero
colunas_percentil = ['inadimplencias_2anos', 'registros_negativos']

for col in colunas_percentil:
    p99 = dados[col].quantile(0.99)
    dados[col] = dados[col].clip(upper=p99)
    print(f"{col} — max após clip: {dados[col].max():.2f}")

print(f"renda_anual — max: {dados['renda_anual'].max():.2f}")
print(f"indice_endividamento — max: {dados['indice_endividamento'].max():.2f}")

In [ ]:
# Análise das colunas com muitos zeros
print("inadimplencias_2anos:")
print(dados['inadimplencias_2anos'].value_counts().head(10))
print(f"% de zeros: {(dados['inadimplencias_2anos']==0).sum()/len(dados)*100:.1f}%")

print("\nregistros_negativos:")
print(dados['registros_negativos'].value_counts().head(10))
print(f"% de zeros: {(dados['registros_negativos']==0).sum()/len(dados)*100:.1f}%")

In [ ]:
# Remover colunas com baixa variabilidade (forte concentração em zero)
dados = dados.drop(columns=['inadimplencias_2anos', 'registros_negativos'])
print(f"Colunas restantes: {dados.columns.tolist()}")

## CRISP-DM Fase 3 — Análise Gráfica dos Dados

In [ ]:
# Distribuição do target
sns.countplot(x='inadimplente', data=dados)
plt.title("Distribuição de Inadimplência")
plt.xlabel("0 = Adimplente | 1 = Inadimplente")
plt.show()

vc = dados['inadimplente'].value_counts()
print(vc)
print(f"\nDesbalanceamento: {vc[0]/vc[1]:.1f}x mais adimplentes")

O dataset apresenta desbalanceamento de ~3.9x entre adimplentes e inadimplentes, o que justifica o uso do SMOTE para equilibrar as classes durante o treinamento.

In [ ]:
# Boxplots das features vs inadimplência
def plot_boxplot_sem_outliers(df, col, target='inadimplente', titulo=None):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    filtrado = df[
        (df[col] >= Q1 - 1.5 * IQR) &
        (df[col] <= Q3 + 1.5 * IQR)
    ]
    sns.boxplot(x=target, y=col, data=filtrado)
    plt.title(titulo or f"{col} vs Inadimplência")
    plt.xlabel("0 = Adimplente | 1 = Inadimplente")
    plt.show()

plot_boxplot_sem_outliers(dados, 'taxa_juros',           titulo='Taxa de Juros vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'score_credito',        titulo='Score de Crédito vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'renda_anual',          titulo='Renda Anual vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'indice_endividamento', titulo='Índice de Endividamento vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'uso_credito_rotativo', titulo='Uso Crédito Rotativo vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'contas_abertas',       titulo='Contas Abertas vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'contas_hipoteca',      titulo='Contas Hipoteca vs Inadimplência')

In [ ]:
# Matriz de correlação
plt.figure(figsize=(10, 8))
sns.heatmap(dados.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de Correlação entre Features')
plt.tight_layout()
plt.show()

print("\nCorrelação com inadimplente (ordenado):")
print(dados.corr()['inadimplente'].sort_values(ascending=False))

In [ ]:
# Remover parcela: multicolinearidade com valor_emprestimo (correlação ~0.95)
dados = dados.drop(columns=['parcela'])
print(f"Colunas após remoção: {dados.columns.tolist()}")
print(f"Shape final: {dados.shape}")

In [ ]:
# Split treino / teste / validação (70/15/15)
X = dados.drop(columns=['inadimplente'])
y = dados['inadimplente']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Treino:    {X_train.shape[0]} registros ({X_train.shape[0]/len(dados)*100:.1f}%)")
print(f"Teste:     {X_test.shape[0]} registros ({X_test.shape[0]/len(dados)*100:.1f}%)")
print(f"Validação: {X_val.shape[0]} registros ({X_val.shape[0]/len(dados)*100:.1f}%)")

## Experimento 1 — Sem SMOTE (baseline)

Todos os modelos usam `StandardScaler` via Pipeline quando necessário, garantindo condições iguais de comparação.

In [ ]:
print("EXPERIMENTO 1 - Sem SMOTE (baseline)")

listaAlgoritmos_exp1 = [
    RandomForestClassifier(n_estimators=100, random_state=42),
    Pipeline([
        ('scaler', StandardScaler()),
        ('modelo', LogisticRegression(max_iter=3000, random_state=42))
    ]),
    GaussianNB(),
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    MLPClassifier(hidden_layer_sizes=(15,), max_iter=1000, random_state=42)
]

listaModelos_exp1 = []
for algoritmo in listaAlgoritmos_exp1:
    algoritmo.fit(X_train, y_train)
    listaModelos_exp1.append(algoritmo)
    print(f"{algoritmo.__class__.__name__} → treinado ✔")

In [ ]:
resultados_exp1 = []

for modelo in listaModelos_exp1:
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    resultados_exp1.append({'modelo': modelo.__class__.__name__, 'acc': acc, 'f1': f1, 'recall': rec})

    print(f"\n{'='*50}")
    print(f"  {modelo.__class__.__name__} (Exp 1)")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Adimplente', 'Inadimplente']))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Adimplente','Inadimplente'],
                yticklabels=['Adimplente','Inadimplente'])
    plt.title(f'Matriz de Confusão — {modelo.__class__.__name__} (Exp 1)')
    plt.ylabel('Real'); plt.xlabel('Previsto')
    plt.tight_layout(); plt.show()

## Experimento 2 — Com SMOTE

SMOTE aplicado apenas ao treino. Teste e validação permanecem com distribuição original.

In [ ]:
print("EXPERIMENTO 2 - Com SMOTE")

sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)
print(f"Antes:  {y_train.value_counts().to_dict()}")
print(f"Depois: {pd.Series(y_train_bal).value_counts().to_dict()}")

# Mesmos algoritmos do Exp 1 — condições iguais para comparação justa
listaAlgoritmos_exp2 = [
    RandomForestClassifier(n_estimators=100, random_state=42),
    Pipeline([
        ('scaler', StandardScaler()),
        ('modelo', LogisticRegression(max_iter=3000, random_state=42))
    ]),
    GaussianNB(),
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    MLPClassifier(hidden_layer_sizes=(15,), max_iter=1000, random_state=42)
]

listaModelos_exp2 = []
for algoritmo in listaAlgoritmos_exp2:
    algoritmo.fit(X_train_bal, y_train_bal)
    listaModelos_exp2.append(algoritmo)
    print(f"{algoritmo.__class__.__name__} → treinado ✔")

In [ ]:
resultados_exp2 = []

for modelo in listaModelos_exp2:
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    resultados_exp2.append({'modelo': modelo.__class__.__name__, 'acc': acc, 'f1': f1, 'recall': rec})

    print(f"\n{'='*50}")
    print(f"  {modelo.__class__.__name__} (Exp 2)")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Adimplente', 'Inadimplente']))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Adimplente','Inadimplente'],
                yticklabels=['Adimplente','Inadimplente'])
    plt.title(f'Matriz de Confusão — {modelo.__class__.__name__} (Exp 2)')
    plt.ylabel('Real'); plt.xlabel('Previsto')
    plt.tight_layout(); plt.show()

melhor_exp2 = listaModelos_exp2[max(range(len(resultados_exp2)), key=lambda i: resultados_exp2[i]['f1'])]
print(f"\nMelhor modelo Exp 2: {melhor_exp2.__class__.__name__}")

## Experimento 3 — VotingClassifier (Ensemble)

In [ ]:
print("EXPERIMENTO 3 - VotingClassifier")

voting = VotingClassifier(
    estimators=[
        ('lr', listaModelos_exp2[1]),  # LogisticRegression
        ('rf', listaModelos_exp2[0]),  # RandomForest
        ('gb', listaModelos_exp2[3])   # GradientBoosting
    ],
    voting='soft'
)

voting.fit(X_train_bal, y_train_bal)
y_pred_voting = voting.predict(X_test)

print(f"\n{'='*50}")
print(f"  VotingClassifier (Exp 3)")
print(f"{'='*50}")
print(classification_report(y_test, y_pred_voting, target_names=['Adimplente', 'Inadimplente']))

cm = confusion_matrix(y_test, y_pred_voting)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Adimplente','Inadimplente'],
            yticklabels=['Adimplente','Inadimplente'])
plt.title('Matriz de Confusão — VotingClassifier (Exp 3)')
plt.ylabel('Real'); plt.xlabel('Previsto')
plt.tight_layout(); plt.show()

acc_voting = accuracy_score(y_test, y_pred_voting)
f1_voting  = f1_score(y_test, y_pred_voting)
rec_voting = recall_score(y_test, y_pred_voting)
print(f"\nAcurácia: {acc_voting:.4f} | F1: {f1_voting:.4f} | Recall: {rec_voting:.4f}")

## Experimento 4 — XGBoost com scale_pos_weight dinâmico

`scale_pos_weight` é calculado dinamicamente a partir dos dados de treino, em vez de fixado em 4.

In [ ]:
print("EXPERIMENTO 4 - XGBoost (scale_pos_weight dinâmico)")

# Calculado dinamicamente — correto independente da amostra
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight calculado: {scale:.2f}")

modelo_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale,  # dinâmico — não hardcoded
    random_state=42,
    eval_metric='logloss'
)

modelo_xgb.fit(X_train, y_train)
y_pred_xgb = modelo_xgb.predict(X_test)

print(f"\n{'='*50}")
print(f"  XGBoost (Exp 4)")
print(f"{'='*50}")
print(classification_report(y_test, y_pred_xgb, target_names=['Adimplente', 'Inadimplente']))

cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Adimplente','Inadimplente'],
            yticklabels=['Adimplente','Inadimplente'])
plt.title('Matriz de Confusão — XGBoost (Exp 4)')
plt.ylabel('Real'); plt.xlabel('Previsto')
plt.tight_layout(); plt.show()

acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb  = f1_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)
print(f"\nAcurácia: {acc_xgb:.4f} | F1: {f1_xgb:.4f} | Recall: {rec_xgb:.4f}")

## Feature Importance — XGBoost

In [ ]:
importances = pd.Series(modelo_xgb.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values()

plt.figure(figsize=(8, 5))
importances_sorted.plot.barh(color='steelblue')
plt.title('Feature Importance — XGBoost')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

print("\nFeature Importance (ordenado):")
print(importances.sort_values(ascending=False))

## Curva ROC — Comparação de Todos os Modelos

In [ ]:
plt.figure(figsize=(9, 6))

modelos_avaliados = [
    ('RandomForest (Exp2)',       listaModelos_exp2[0]),
    ('LogisticRegression (Exp2)', listaModelos_exp2[1]),
    ('GaussianNB (Exp2)',         listaModelos_exp2[2]),
    ('GradientBoosting (Exp2)',   listaModelos_exp2[3]),
    ('VotingClassifier (Exp3)',   voting),
    ('XGBoost (Exp4)',            modelo_xgb),
]

for nome, modelo in modelos_avaliados:
    proba = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f"{nome} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Aleatório')
plt.xlabel('FPR (Falso Positivo)')
plt.ylabel('TPR (Recall)')
plt.title('Curva ROC — Comparação dos Modelos')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

## Comparação Final dos Experimentos

In [ ]:
print(f"{'='*70}")
print(f"{'Experimento':<40} {'Acurácia':<12} {'F1 Inad.':<12} {'Recall Inad.'}")
print(f"{'='*70}")

for r in resultados_exp1:
    print(f"Exp1 {r['modelo']:<35} {r['acc']:.4f}       {r['f1']:.4f}       {r['recall']:.4f}")

print(f"{'-'*70}")

for r in resultados_exp2:
    print(f"Exp2 {r['modelo']:<35} {r['acc']:.4f}       {r['f1']:.4f}       {r['recall']:.4f}")

print(f"{'-'*70}")
print(f"Exp3 {'VotingClassifier':<35} {acc_voting:.4f}       {f1_voting:.4f}       {rec_voting:.4f}")

print(f"{'-'*70}")
print(f"Exp4 {'XGBoost':<35} {acc_xgb:.4f}       {f1_xgb:.4f}       {rec_xgb:.4f}")

print(f"{'='*70}")
print(f"\n✔ Melhor modelo: XGBoost (Exp 4)")
print(f"  F1: {f1_xgb:.4f} | Recall: {rec_xgb:.4f}")

In [ ]:
# Salvar melhor modelo
NOMEMODELO = 'modelo_inadimplencia_v2.pickle'

with open(NOMEMODELO, 'wb') as f:
    pickle.dump(modelo_xgb, f)
print(f"Modelo salvo como '{NOMEMODELO}' ✔")

# Validar no conjunto de holdout
with open(NOMEMODELO, 'rb') as f:
    modelo_carregado = pickle.load(f)

y_pred_val = modelo_carregado.predict(X_val)
print(f"\nAcurácia na validação: {accuracy_score(y_val, y_pred_val):.4f}")
print(classification_report(y_val, y_pred_val, target_names=['Adimplente', 'Inadimplente']))

# Exemplo em produção
print("\nExemplo de predição em produção:")
novo_cliente = pd.DataFrame([X_val.iloc[0]], columns=X_val.columns)
proba_cliente = modelo_carregado.predict_proba(novo_cliente)[0][1]
pred = modelo_carregado.predict(novo_cliente)[0]
print(f"Probabilidade de inadimplência: {proba_cliente:.2%}")
print(f"Predição: {'Inadimplente ⚠️' if pred == 1 else 'Adimplente ✔'}")

# Conclusão Final do Projeto

## Dataset
- **Fonte:** Lending Club (2007–2018)
- **Registros:** ~114 mil
- **Features:** 9 (após remoção de `parcela` por multicolinearidade)
- **Target:** `inadimplente` (0 = Adimplente, 1 = Inadimplente)

---

## Experimentos Realizados

| Experimento | Descrição | Melhor Modelo | F1-score |
|------------|----------|--------------|----------|
| Exp 1 | Sem SMOTE (baseline) | GaussianNB | ~0.35 |
| Exp 2 | Com SMOTE | LogisticRegression | ~0.42 |
| Exp 3 | Voting Classifier | Ensemble | ~0.36 |
| **Exp 4** | **XGBoost (scale_pos_weight dinâmico)** | **XGBoost** | **~0.43** |

---

## Modelo Escolhido: XGBoost

- **Acurácia (validação):** ~0.66
- **F1-score (inadimplente):** ~0.43
- **Recall (inadimplente):** ~0.64

---

## Principais Conclusões

- **SMOTE foi decisivo** para modelos tradicionais: Recall aumentou de ~0.09 para ~0.62
- **XGBoost apresentou o melhor desempenho geral**, mesmo sem SMOTE, usando `scale_pos_weight` dinâmico
- **VotingClassifier não superou** modelos individuais
- **F1-score de ~0.43 é consistente** com dados reais de crédito, onde a previsibilidade é naturalmente limitada

## Interpretação de Negócio

Em problemas de crédito, os erros têm **custos assimétricos**:
- ⚠️ **Falso Negativo** (inadimplente classificado como adimplente) → alto custo (perda financeira)
- **Falso Positivo** (adimplente classificado como inadimplente) → custo menor (oportunidade perdida)

Por isso, o modelo foi escolhido priorizando o **Recall da classe inadimplente**, reduzindo o risco de concessão de crédito inadequada.